# Train RSGM on the AMASS of PoseD-Flow

RSGM (`hp/models/rsgm.py`) trained on **their** data, for the fair comparison with their pretrained PoseD-Flow in `explore/evaluation/posedflow_vs_rsgm.ipynb`: same data, same split, same body model.

- **Data:** their AMASS (the preprocessing of DPoser: SMPL fits, 30 Hz, the first and last 10% of each sequence cut), converted by `scripts/preprocess/convert_posedflow.py` into `~/data/AMASS_rotations_posedflow_c10`:
  - train: every 10th of their 18.6M poses (1.86M; neighbouring frames are nearly the same);
  - valid: every 100th (8,601), as their own training validates;
  - test: untouched here.

  Our 19 joints are their 21 without the toes.
- **Hyperparameters:** not tuned for this data. They are rounded values from the middle of the plateau of the last Optuna search on our AMASS (`scripts/tune_rsgm.py`, every 20th frame, 30k steps). Its 15 best trials all lie within 0.5 nats of each other, about the noise between seeds, and agree on:
  - batch 4096, width 1536, EMA 0.999, step schedule, std weighting;
  - depth 5–8, lr 1.3–2.4e-3, β_f 5.6–8.7, loss_t_min 1–4e-3.

  The point is a configuration that transfers, not one fitted to this data. Weight decay had no visible effect (1e-7 … 7e-5).
- **Budget:** 30k steps, as in the search, whose trials peaked at 20–25k (the step schedule halves the rate at 70%, i.e. 21k). On 1.86M poses this is 66 epochs. The checkpoints with the lowest validation loss are kept (`keep_best`).
- **Size:** 12M parameters against about 0.7M of PoseD-Flow (4 × 512). `CONFIG = "small"` trains a 4 × 512 network with the same settings as a control for size.
- **Seeds:** one run per seed in `SEEDS`, each in its own folder `rsgm_posedflow[_small]_s<seed>`. A finished run is not trained again.

In [ ]:
import json
import sys
import time
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

from hp.evaluation.data import load_reference
from hp.models.io import load_model
from hp.training import cli
from hp.training.config import load_config

assert cli.__file__.startswith(SRC), f"hp comes from {cli.__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"
SPLIT = "posedflow"
RUN_ROOT = "~/data/experiments"
CONFIG = "large"                           # "search": the rounded configuration; "small": 4 x 512, a control for size, "large" - new model to train
SEEDS = [10]                                # e.g. [10, 11, 12] for the spread between seeds
DEVICE = "mps"                              # "cpu", "mps" or "cuda:0"
RETRAIN = False                             # True: train a new run even if a finished one exists
N_CHECK = 500                               # valid poses for the quick likelihood check at the end

COMMON = [  # rounded from the plateau of the search (see above)
    "model.args.activation=silu",
    "model.args.beta_f=7.5",
    "model.args.loss_t_min=0.002",
    "model.args.weighting=std",
    "train.lr=0.002",
    "train.batch_size=4096",
    "train.ema_decay=0.999",
    "train.scheduler=step",
    "train.optimizer=adamw",
    "train.weight_decay=1e-5",
    "train.max_iterations=30000",
    "train.valid_interval=1000",
    "train.checkpoint_every=5000",
]
CONFIGS = {
    "search": ["model.args.hidden=[1536, 1536, 1536, 1536, 1536, 1536]"],
    "small": ["model.args.hidden=[512, 512, 512, 512]"],  # the size of PoseD-Flow's network
    "large": [
        "model.args.hidden=[4096, 4096, 4096, 4096, 4096, 4096]",
        "train.lr=1e-3",
        "train.max_iterations=30000",
    ],
}
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
name_of = lambda seed: f"rsgm_posedflow{'' if CONFIG == 'search' else '_' + CONFIG}_s{seed}"
argv_of = lambda seed: ["model=rsgm", f"run_root={RUN_ROOT}", f"run_name={name_of(seed)}", f"seed={seed}",
                        f"data.rotations={ROTATIONS}", f"data.split={SPLIT}", "data.load_dtype=float32",
                        f"train.device={DEVICE}", *COMMON, *CONFIGS[CONFIG]]
cfg = load_config(argv_of(SEEDS[0]))
print(OmegaConf.to_yaml(cfg))

train_frames = torch.load(expand(ROTATIONS) / "PoseDFlow_train.pt", mmap=True, weights_only=True).shape[0]
iterations, batch = cfg.train.max_iterations, cfg.train.batch_size
print(f"{iterations:,} iterations x batch {batch} = {iterations * batch / train_frames:.1f} epochs over {train_frames:,} training poses")

## Train

In [ ]:
def finished_run(root):
    # The newest run folder below root that finished (has history.json and weights.pt), or None.
    runs = sorted(p for p in root.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / "weights.pt").is_file())
    return runs[-1] if runs else None


run_dirs = {}
for seed in SEEDS:
    run_dir = None if RETRAIN else finished_run(expand(RUN_ROOT) / name_of(seed))
    if run_dir is not None:
        print(f"seed {seed}: already trained, {run_dir}")
    else:
        start = time.perf_counter()
        run_dir = cli.main(argv_of(seed))
        seconds = time.perf_counter() - start
        (run_dir / "train_time.json").write_text(json.dumps({"seconds": seconds, "device": DEVICE}))
        print(f"seed {seed}: trained in {seconds / 60:.1f} min, {run_dir}")
    run_dirs[seed] = run_dir

## Loss curves


Training loss (moving average) and validation loss (EMA weights, fixed times and noise) of every seed.

In [ ]:
SMOOTH = 200  # steps of the moving average

fig, ax = plt.subplots(figsize=(10, 4))
for k, (seed, run_dir) in enumerate(run_dirs.items()):
    history = json.loads((run_dir / "history.json").read_text())
    loss = np.asarray(history["loss"], dtype=float)
    window = min(SMOOTH, len(loss))
    smooth = np.convolve(loss, np.ones(window) / window, mode="valid")
    ax.plot(np.arange(window, len(loss) + 1), smooth, color=f"C{k}", alpha=0.5, label=f"seed {seed}: training")
    ax.plot(history["valid_iteration"], history["valid_loss"], "o-", ms=3, color=f"C{k}", label=f"seed {seed}: validation")
    if history["valid_loss"]:
        best = int(np.argmin(history["valid_loss"]))
        print(f"seed {seed}: validation loss last {history['valid_loss'][-1]:.3f}, "
              f"best {history['valid_loss'][best]:.3f} at step {history['valid_iteration'][best]:,}")
ax.set_xlabel("step")
ax.set_ylabel("denoising score matching loss")
ax.set_yscale("log")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Quick check

The EMA weights of the last step: the samples must be valid rotations, and the log-likelihood of `N_CHECK` valid poses must be finite. The comparison with PoseD-Flow is in `explore/evaluation/posedflow_vs_rsgm.ipynb` (`RSGM_RUN = "~/data/experiments/rsgm_posedflow_s10"`).

In [ ]:
valid = load_reference("valid", ROTATIONS, SPLIT)
for seed, run_dir in run_dirs.items():
    model = load_model(run_dir, device=DEVICE, weights="weights_ema")
    torch.manual_seed(seed)
    samples = model.sample(1000).cpu().double()
    eye = torch.eye(3, dtype=torch.float64)
    poses = valid[torch.randperm(len(valid), generator=torch.Generator().manual_seed(seed))[:N_CHECK]]
    with torch.no_grad():
        ll = torch.cat([model.log_prob(poses[i:i + 250].to(model.device)).cpu() for i in range(0, len(poses), 250)])
    print(f"seed {seed}: max |R^T R - I| = {(samples.transpose(-1, -2) @ samples - eye).abs().max():.1e}, "
          f"valid log-likelihood {ll.mean():.2f} +- {ll.std() / len(ll) ** 0.5:.2f} (s.e.m.), {(~torch.isfinite(ll)).sum()} non-finite")